# Мини-проект по задаче кредитного скоринга

## Выбор моделей

Для решения задачи кредитного скоринга были выбраны несколько моделей классификации, позволяющих сравнить разные подходы к обучению:

- **Logistic Regression** — базовая линейная модель для бинарной классификации;
- **K-Nearest Neighbors** — метод, основанный на близости объектов в пространстве признаков;
- **Decision Tree Classifier** — интерпретируемая нелинейная модель;
- **Random Forest Classifier** — ансамблевая модель на основе множества деревьев решений.

Такой набор моделей позволяет сравнить линейный подход, метрический алгоритм, одиночную древовидную модель и ансамбль деревьев.

## Все необходимые импорты

In [42]:
import numpy as np
import pandas as pd

%matplotlib inline
from matplotlib import pyplot as plt

from sklearn.model_selection import (
    train_test_split,
    cross_val_score,
    GridSearchCV,
    StratifiedKFold
)
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    accuracy_score,
    roc_auc_score
)

## Работа с датафреймом
Создание тестовых и отложенных выборок

In [43]:
bank_data = pd.read_csv('../data/bank_datafile.csv')

In [44]:
bank_data.head()

,ID,LIMIT_BAL,SEX,EDUCATION,MARRIAGE,AGE,PAY_0,PAY_2,PAY_3,PAY_4,...,BILL_AMT4,BILL_AMT5,BILL_AMT6,PAY_AMT1,PAY_AMT2,PAY_AMT3,PAY_AMT4,PAY_AMT5,PAY_AMT6,default payment next month
0,1,20000,2,2,1,24,2,2,-1,-1,...,0,0,0,0,689,0,0,0,0,1
1,2,120000,2,2,2,26,-1,2,0,0,...,3272,3455,3261,0,1000,1000,1000,0,2000,1
2,3,90000,2,2,2,34,0,0,0,0,...,14331,14948,15549,1518,1500,1000,1000,1000,5000,0
3,4,50000,2,2,1,37,0,0,0,0,...,28314,28959,29547,2000,2019,1200,1100,1069,1000,0
4,5,50000,1,2,1,57,-1,0,-1,0,...,20940,19146,19131,2000,36681,10000,9000,689,679,0


In [45]:
y = bank_data['default payment next month']
X = bank_data.drop(['default payment next month'], axis=1)
X_train, X_holdout, y_train, y_holdout = train_test_split(X, y, test_size=0.3, random_state=17, stratify=y)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=17)

In [46]:
bank_data.shape
bank_data.info()
bank_data.isna().sum().sort_values(ascending=False).head(10)
y.value_counts()
y.value_counts(normalize=True)
bank_data.describe()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 30000 entries, 0 to 29999
Data columns (total 25 columns):
 #   Column                      Non-Null Count  Dtype
---  ------                      --------------  -----
 0   ID                          30000 non-null  int64
 1   LIMIT_BAL                   30000 non-null  int64
 2   SEX                         30000 non-null  int64
 3   EDUCATION                   30000 non-null  int64
 4   MARRIAGE                    30000 non-null  int64
 5   AGE                         30000 non-null  int64
 6   PAY_0                       30000 non-null  int64
 7   PAY_2                       30000 non-null  int64
 8   PAY_3                       30000 non-null  int64
 9   PAY_4                       30000 non-null  int64
 10  PAY_5                       30000 non-null  int64
 11  PAY_6                       30000 non-null  int64
 12  BILL_AMT1                   30000 non-null  int64
 13  BILL_AMT2                   30000 non-null  int64
 14  BILL_A

,ID,LIMIT_BAL,SEX,EDUCATION,MARRIAGE,AGE,PAY_0,PAY_2,PAY_3,PAY_4,...,BILL_AMT4,BILL_AMT5,BILL_AMT6,PAY_AMT1,PAY_AMT2,PAY_AMT3,PAY_AMT4,PAY_AMT5,PAY_AMT6,default payment next month
count,30000.000000,30000.000000,30000.000000,30000.000000,30000.000000,30000.000000,30000.000000,30000.000000,30000.000000,30000.000000,...,30000.000000,30000.000000,30000.000000,30000.000000,3.000000e+04,30000.00000,30000.000000,30000.000000,30000.000000,30000.000000
mean,15000.500000,167484.322667,1.603733,1.853133,1.551867,35.485500,-0.016700,-0.133767,-0.166200,-0.220667,...,43262.948967,40311.400967,38871.760400,5663.580500,5.921163e+03,5225.68150,4826.076867,4799.387633,5215.502567,0.221200
std,8660.398374,129747.661567,0.489129,0.790349,0.521970,9.217904,1.123802,1.197186,1.196868,1.169139,...,64332.856134,60797.155770,59554.107537,16563.280354,2.304087e+04,17606.96147,15666.159744,15278.305679,17777.465775,0.415062
min,1.000000,10000.000000,1.000000,0.000000,0.000000,21.000000,-2.000000,-2.000000,-2.000000,-2.000000,...,-170000.000000,-81334.000000,-339603.000000,0.000000,0.000000e+00,0.00000,0.000000,0.000000,0.000000,0.000000
25%,7500.750000,50000.000000,1.000000,1.000000,1.000000,28.000000,-1.000000,-1.000000,-1.000000,-1.000000,...,2326.750000,1763.000000,1256.000000,1000.000000,8.330000e+02,390.00000,296.000000,252.500000,117.750000,0.000000
50%,15000.500000,140000.000000,2.000000,2.000000,2.000000,34.000000,0.000000,0.000000,0.000000,0.000000,...,19052.000000,18104.500000,17071.000000,2100.000000,2.009000e+03,1800.00000,1500.000000,1500.000000,1500.000000,0.000000
75%,22500.250000,240000.000000,2.000000,2.000000,2.000000,41.000000,0.000000,0.000000,0.000000,0.000000,...,54506.000000,50190.500000,49198.250000,5006.000000,5.000000e+03,4505.00000,4013.250000,4031.500000,4000.000000,0.000000
max,30000.000000,1000000.000000,2.000000,6.000000,3.000000,79.000000,8.000000,8.000000,8.000000,8.000000,...,891586.000000,927171.000000,961664.000000,873552.000000,1.684259e+06,896040.00000,621000.000000,426529.000000,528666.000000,1.000000


## Логическая регрессия

In [47]:
logres_pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('model', LogisticRegression(random_state=17))
])

In [48]:
logres_results = cross_val_score(logres_pipe, X_train, y_train, n_jobs=-1, scoring='roc_auc')

In [49]:
print(f'LogisticRegression cross_val_score: {logres_results.mean()}')

LogisticRegression cross_val_score: 0.7187186611376551


## Подбор гиперпараметров для логистической регрессии

Для логистической регрессии выполняется подбор коэффициента регуляризации `C` и параметра `class_weight`.

Параметр `C` управляет силой регуляризации модели.
Параметр `class_weight` позволяет проверить, улучшает ли учет дисбаланса классов качество классификации.

Так как логистическая регрессия чувствительна к масштабу признаков, обучение выполняется в `Pipeline` вместе с `StandardScaler`.

In [50]:
logreg_params = {
    'model__C': [0.001, 0.01, 0.1, 1, 10, 100],
    'model__class_weight': [None, 'balanced']
}
best_logres = GridSearchCV(logres_pipe, logreg_params, n_jobs=-1, scoring='roc_auc')
best_logres.fit(X_train,y_train)

GridSearchCV(estimator=Pipeline(steps=[('scaler', StandardScaler()),
                                       ('model',
                                        LogisticRegression(random_state=17))]),
             n_jobs=-1,
             param_grid={'model__C': [0.001, 0.01, 0.1, 1, 10, 100],
                         'model__class_weight': [None, 'balanced']},
             scoring='roc_auc')

In [51]:
print('Best params:', best_logres.best_params_)
print('Best CV ROC-AUC:', best_logres.best_score_)

Best params: {'model__C': 10, 'model__class_weight': 'balanced'}
Best CV ROC-AUC: 0.719240189827492


## Бинарное дерево

In [52]:
tree = DecisionTreeClassifier(random_state=17)

In [53]:
tree_results = cross_val_score(tree, X_train, y_train, n_jobs=-1, scoring='roc_auc')

Результаты работы `пустого` дерева

In [54]:
print(f'Tree cross_val_score: {tree_results.mean()}')

Tree cross_val_score: 0.6134818193874538


## Подбор гиперпараметров для дерева решений

Для уменьшения переобучения и поиска более устойчивой модели выполняется подбор гиперпараметров `DecisionTreeClassifier` с помощью `GridSearchCV`.

В ходе подбора анализируются параметры, влияющие на сложность дерева:
- `max_depth` — максимальная глубина дерева;
- `min_samples_split` — минимальное число объектов для разбиения узла;
- `min_samples_leaf` — минимальное число объектов в листе;
- `criterion` — функция оценки качества разбиения.

In [55]:
tree_params = {    
    'criterion': ['gini', 'entropy', 'log_loss'],
    'max_depth': [3, 4, 5, 6, 8, 10, 12],
    'min_samples_split': [20, 50, 100, 200],
    'min_samples_leaf': [5, 10, 20, 50]}
best_tree = GridSearchCV(tree, tree_params, n_jobs=-1, scoring='roc_auc')
best_tree.fit(X_train,y_train)

GridSearchCV(estimator=DecisionTreeClassifier(random_state=17), n_jobs=-1,
             param_grid={'criterion': ['gini', 'entropy', 'log_loss'],
                         'max_depth': [3, 4, 5, 6, 8, 10, 12],
                         'min_samples_leaf': [5, 10, 20, 50],
                         'min_samples_split': [20, 50, 100, 200]},
             scoring='roc_auc')

In [56]:
print('Best params:', best_tree.best_params_)
print('Best CV ROC-AUC:', best_tree.best_score_)

Best params: {'criterion': 'gini', 'max_depth': 10, 'min_samples_leaf': 50, 'min_samples_split': 200}
Best CV ROC-AUC: 0.760983085529323


## K-Nearest Neighbors

In [57]:
knn_pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('model', KNeighborsClassifier())
])

In [58]:
knn_results = cross_val_score(knn_pipe, X_train, y_train, n_jobs=-1, scoring='roc_auc')

In [59]:
print(f'KNN cross_val_score: {knn_results.mean()}')

KNN cross_val_score: 0.695076575667896


## Подбор гиперпараметров для KNN

В качестве одной из моделей классификации рассматривается метод ближайших соседей (`K-Nearest Neighbors`, KNN).

Данный алгоритм относит объект к классу на основе классов его ближайших соседей в пространстве признаков.  
Так как KNN использует расстояния между объектами, перед обучением признаки масштабируются с помощью `StandardScaler`.

Для подбора оптимальной конфигурации модели используются следующие гиперпараметры:
- `n_neighbors` — количество ближайших соседей;
- `weights` — способ взвешивания соседей;
- `p` — параметр метрики Минковского, определяющий тип расстояния.

In [60]:
knn_params = {
    'model__n_neighbors': [3, 5, 7, 9, 11, 15, 21, 28],
    'model__weights': ['uniform', 'distance'],
    'model__metric': ['minkowski'],
    'model__p': [1, 2]
}
best_knn = GridSearchCV(knn_pipe, knn_params, n_jobs=-1, scoring='roc_auc')
best_knn.fit(X_train, y_train)

GridSearchCV(estimator=Pipeline(steps=[('scaler', StandardScaler()),
                                       ('model', KNeighborsClassifier())]),
             n_jobs=-1,
             param_grid={'model__metric': ['minkowski'],
                         'model__n_neighbors': [3, 5, 7, 9, 11, 15, 21, 28],
                         'model__p': [1, 2],
                         'model__weights': ['uniform', 'distance']},
             scoring='roc_auc')

In [61]:
print('Best params:', best_knn.best_params_)
print('Best CV ROC-AUC:', best_knn.best_score_)

Best params: {'model__metric': 'minkowski', 'model__n_neighbors': 28, 'model__p': 1, 'model__weights': 'distance'}
Best CV ROC-AUC: 0.7429741220017776


## Случайный лес

In [62]:
forest = RandomForestClassifier(random_state=17)

In [63]:
forest_results = cross_val_score(forest, X_train, y_train, n_jobs=-1, scoring='roc_auc')

In [64]:
print(f'RandomForest cross_val_score: {forest_results.mean()}')

RandomForest cross_val_score: 0.766289231887096


## Подбор гиперпараметров для случайного леса

Случайный лес представляет собой набор деревьев решений, обучающихся на различных подвыборках данных, а итоговое предсказание формируется на основе агрегированного результата всех деревьев. Такой подход позволяет уменьшить переобучение по сравнению с одиночным деревом решений и повысить устойчивость модели.

В ходе подбора гиперпараметров анализируются:
- `n_estimators` — количество деревьев в ансамбле;
- `max_depth` — максимальная глубина деревьев;
- `min_samples_split` — минимальное число объектов для разбиения узла;
- `min_samples_leaf` — минимальное число объектов в листе;
- `max_features` — число признаков, рассматриваемых при поиске лучшего разбиения;

Подбор гиперпараметров выполняется с помощью `GridSearchCV`, а основной метрикой качества выступает `ROC-AUC`.

In [65]:
rf_params = {
    'n_estimators': [100, 200],
    'max_depth': [8, 12, None],
    'min_samples_split': [2, 20],
    'min_samples_leaf': [1, 5],
    'max_features': ['sqrt', 'log2']
}
best_forest = GridSearchCV(forest, rf_params, n_jobs=-1, scoring='roc_auc')
best_forest.fit(X_train, y_train)

GridSearchCV(estimator=RandomForestClassifier(random_state=17), n_jobs=-1,
             param_grid={'max_depth': [8, 12, None],
                         'max_features': ['sqrt', 'log2'],
                         'min_samples_leaf': [1, 5],
                         'min_samples_split': [2, 20],
                         'n_estimators': [100, 200]},
             scoring='roc_auc')

In [66]:
print('Best params:', best_forest.best_params_)
print('Best CV ROC-AUC:', best_forest.best_score_)

Best params: {'max_depth': 12, 'max_features': 'sqrt', 'min_samples_leaf': 5, 'min_samples_split': 20, 'n_estimators': 200}
Best CV ROC-AUC: 0.7800869368054525


# Результаты

In [67]:
results = pd.DataFrame({
    'Model': [
        'Logistic Regression',
        'Decision Tree',
        'KNN',
        'Random Forest'
    ],
    'CV ROC-AUC': [
        best_logres.best_score_,
        best_tree.best_score_,
        best_knn.best_score_,
        best_forest.best_score_
    ],
    'Holdout ROC-AUC': [
        roc_auc_score(y_holdout, best_logres.predict_proba(X_holdout)[:, 1]),
        roc_auc_score(y_holdout, best_tree.predict_proba(X_holdout)[:, 1]),
        roc_auc_score(y_holdout, best_knn.predict_proba(X_holdout)[:, 1]),
        roc_auc_score(y_holdout, best_forest.predict_proba(X_holdout)[:, 1])
    ]
})

results.sort_values(by='Holdout ROC-AUC', ascending=False)

,Model,CV ROC-AUC,Holdout ROC-AUC
3,Random Forest,0.780087,0.782981
1,Decision Tree,0.760983,0.765998
2,KNN,0.742974,0.752532
0,Logistic Regression,0.719240,0.728864


## Итоговые выводы

В ходе работы были рассмотрены четыре модели классификации: Logistic Regression, Decision Tree, K-Nearest Neighbors и Random Forest.

Наилучший результат на отложенной выборке показала модель Random Forest, что говорит о преимуществе ансамблевого подхода в данной задаче. Логистическая регрессия показала более простой, но устойчивый уровень. KNN оказался чувствительным к масштабу признаков, а одиночное дерево решений уступило случайному лесу по качеству и устойчивости.

Таким образом, для задачи кредитного скоринга наиболее подходящей из рассмотренных моделей оказалась Random Forest Classifier.